# Per-frame independent fitting (no OLA overlap)

For each control frame `t ∈ [0, T)`, run `B` independent optimizations in parallel on **just that frame's audio chunk**, take the per-frame argmin (across `B` and across optimization steps), and stitch the winners into a full trajectory.

This is the embarrassingly-parallel cousin of `0425_tract_fit_constrained.ipynb`: a flat `T × B` batch instead of `B` joint fits over `T` frames.

**No-overlap synth:** `pink_trombone_ola` is called with `params.shape = [T*B, 1, 13]` so each batch element gets a single-frame waveguide IR convolved against a single-hop glottis source — no IR tail bleeds into a "next" frame, because there is no next frame in that call. The loss compares each predicted frame chunk to the corresponding target chunk.

**Frame size:** `samples_per_frame = 4096` (`control_rate ≈ 10.77 Hz`) so the loss STFT can use `n_fft = hop = 4096, center=False` — exactly one spectrum per chunk, no padding leakage.

**Caveats:**
- Per-frame synth resets glottis phase / vibrato / simplex noise to 0 at the start of every frame. Fine for the (phase-invariant) STFT loss.
- The stitched resynth uses the continuous-state `pink_trombone_ola` so phase is glued back across frames for listening — what you hear is slightly different from what the optimizer scored.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone_ola,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load and chunk target audio

Frame size is set locally so `n_fft = samples_per_frame` works cleanly for `torch.stft(center=False)`.

In [ ]:
from samuel.data import _load_resampled, load_manifest

MANIFEST_PATH = (Path(".") / "manifests" / "librilight_10h.jsonl").resolve()
CLIP_INDEX = 0
START_OFFSET = 1.0
TARGET_DURATION = 2.0

# Power-of-two frame size: lets us use n_fft = hop_length for the loss with no padding.
SAMPLES_PER_FRAME = 1024
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME  # ≈ 10.77 Hz

files = load_manifest(MANIFEST_PATH)
df = files[CLIP_INDEX]
print(f"manifest: {len(files)} files")
print(f"using: {df.path.name} ({df.duration:.1f}s @ {df.sample_rate} Hz)")

audio = _load_resampled(df.path, SAMPLE_RATE)
start = int(round(START_OFFSET * SAMPLE_RATE))
n_target = int(round(TARGET_DURATION * SAMPLE_RATE))
assert start + n_target <= len(audio), f"file too short: {len(audio)/SAMPLE_RATE:.1f}s"
wav_rs = audio[start : start + n_target].astype(np.float32)

T = wav_rs.shape[0] // SAMPLES_PER_FRAME
T_samples = T * SAMPLES_PER_FRAME
wav_rs = wav_rs[:T_samples]
target = torch.from_numpy(wav_rs).unsqueeze(0).to(device)
target_frames = target.view(T, SAMPLES_PER_FRAME)  # [T, samples_per_frame]
print(
    f"target: {tuple(target.shape)}, T={T} frames @ {CONTROL_RATE:.3f} Hz "
    f"({T_samples / SAMPLE_RATE:.2f} s)"
)

print("\nTarget:")
display(Audio(target[0].cpu().numpy(), rate=SAMPLE_RATE))

## 2. Constrained per-frame parameterization

`raw [T, B, n_trainable]`: each `(t, b)` is one independent fit at frame `t` from random init `b`. The `frequency` column is overridden with the pyin-derived f0 trajectory (broadcast over `B`).

In [ ]:
PARAM_SPEC = {
    "noise":                (0.0, 0.5),
    "frequency":            (80.0, 400.0),
    "tenseness":            (0.0, 1.0),
    "intensity":            (0.0, 1.0),
    "loudness":             (0.0, 1.0),
    "tongueIndex":          (10.0, 35.0),
    "tongueDiameter":       (1.5, 3.5),
    "vibratoWobble":        None,   # frozen at 0
    "vibratoFrequency":     None,   # frozen at 6
    "vibratoGain":          None,   # frozen at 0
    "tractLength":          None,   # frozen at 44
    "constrictionIndex":    (0.0, 44.0),
    "constrictionDiameter": (-0.5, 3.0),
}
FROZEN_VALUES = {
    "vibratoWobble": 0.0,
    "vibratoFrequency": 6.0,
    "vibratoGain": 0.0,
    "tractLength": 44.0,
}

trainable_names = [n for n in PARAM_NAMES if PARAM_SPEC[n] is not None]
n_trainable = len(trainable_names)

B = 64
print(f"trainable: {n_trainable} params x T={T} frames x B={B} = {n_trainable * T * B} scalars")

_lo = torch.tensor([PARAM_SPEC[n][0] for n in trainable_names], device=device)
_hi = torch.tensor([PARAM_SPEC[n][1] for n in trainable_names], device=device)

# pyin-derived f0 trajectory at the local control rate. pyin runs at a fine
# hop and we resample to the T frame centres; unvoiced frames get the nearest
# voiced neighbour via np.interp.
import librosa

FREQ_LO, FREQ_HI = PARAM_SPEC["frequency"]
PYIN_HOP = 512
f0_pyin, _, _ = librosa.pyin(
    target[0].cpu().numpy().astype(np.float32),
    sr=SAMPLE_RATE,
    fmin=FREQ_LO,
    fmax=FREQ_HI,
    frame_length=2048,
    hop_length=PYIN_HOP,
)
t_pyin = librosa.times_like(f0_pyin, sr=SAMPLE_RATE, hop_length=PYIN_HOP)
t_ctrl = (np.arange(T) + 0.5) * SAMPLES_PER_FRAME / SAMPLE_RATE

voiced_mask = np.isfinite(f0_pyin)
if voiced_mask.any():
    f0_init = np.interp(t_ctrl, t_pyin[voiced_mask], f0_pyin[voiced_mask]).astype(np.float32)
else:
    f0_init = np.full(T, 0.5 * (FREQ_LO + FREQ_HI), dtype=np.float32)
f0_init = np.clip(f0_init, FREQ_LO + 1e-3, FREQ_HI - 1e-3)
print(
    f"pyin: {int(voiced_mask.sum())}/{voiced_mask.size} voiced; "
    f"resampled to T={T}; f0 range [{f0_init.min():.1f}, {f0_init.max():.1f}] Hz"
)

# raw [T, B, n_trainable]: random per (t, b) in normalized [0.1, 0.9].
_gen = torch.Generator(device=device).manual_seed(42)
_init_norm = torch.rand(T, B, n_trainable, generator=_gen, device=device) * 0.8 + 0.1
raw = torch.log(_init_norm / (1 - _init_norm))

# Override frequency column with pyin trajectory (same across B).
freq_col = trainable_names.index("frequency")
f0_norm = (torch.from_numpy(f0_init).to(device) - FREQ_LO) / (FREQ_HI - FREQ_LO)
f0_norm = f0_norm.clamp(1e-4, 1 - 1e-4)
raw[:, :, freq_col] = torch.log(f0_norm / (1 - f0_norm)).unsqueeze(1)  # [T, 1] -> [T, B]

raw.requires_grad_(True)

def to_synth_params(raw: torch.Tensor) -> torch.Tensor:
    """raw [..., n_trainable] -> synth params [..., 13] in valid ranges."""
    constrained = torch.sigmoid(raw) * (_hi - _lo) + _lo
    out = raw.new_zeros(*raw.shape[:-1], N_PARAMS)
    for j, name in enumerate(trainable_names):
        i = PARAM_NAMES.index(name)
        out[..., i] = constrained[..., j]
    for name, val in FROZEN_VALUES.items():
        i = PARAM_NAMES.index(name)
        out[..., i] = val
    return out

with torch.no_grad():
    p0 = to_synth_params(raw)[0, 0]  # frame 0, init 0
    print("\nframe 0, init 0:")
    for name in PARAM_NAMES:
        print(f"  {name:22s} = {p0[PARAM_NAMES.index(name)].item():.3f}")

## 3. Single-scale STFT loss (window = frame size)

`n_fft = hop = samples_per_frame`, `center=False` → exactly one spectrum per chunk, no padding.

In [ ]:
N_FFT = SAMPLES_PER_FRAME
_loss_window = torch.hann_window(N_FFT, device=device)

def per_frame_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Both [N, samples_per_frame]; returns [N]."""
    s_p = torch.stft(
        pred, N_FFT, N_FFT, window=_loss_window, center=False, return_complex=True
    ).abs()
    s_t = torch.stft(
        target, N_FFT, N_FFT, window=_loss_window, center=False, return_complex=True
    ).abs()
    return (torch.log1p(s_p) - torch.log1p(s_t)).abs().mean(dim=(-1, -2))

# tgt_rep is reused every step. [T, B, samples_per_frame] flat -> [T*B, samples_per_frame].
tgt_rep = target_frames.unsqueeze(1).expand(T, B, SAMPLES_PER_FRAME).reshape(T * B, SAMPLES_PER_FRAME).contiguous()

with torch.no_grad():
    raw_flat = raw.detach().reshape(T * B, 1, n_trainable)
    init_pred = pink_trombone_ola(
        to_synth_params(raw_flat), seed=0, control_rate=CONTROL_RATE
    )  # [T*B, samples_per_frame]
    assert init_pred.shape == (T * B, SAMPLES_PER_FRAME), init_pred.shape
    init_losses = per_frame_loss(init_pred, tgt_rep).reshape(T, B)
    print(
        f"initial losses: shape {tuple(init_losses.shape)}, "
        f"min={init_losses.min().item():.4f}, "
        f"per-frame argmin mean={init_losses.min(dim=1).values.mean().item():.4f}"
    )

## 4. Training loop

One Adam step per outer iteration: `raw [T, B, n_trainable]` flatten → synth `[T*B, 1, 13]` → audio `[T*B, hop]` → per-element STFT loss `[T*B]` → mean. Per-frame best (across B and across steps) is tracked on the side.

In [ ]:
import time
from tqdm.auto import tqdm

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

N_STEPS = 50
LR = 0.05
IR_LENGTH = 256  # short IR keeps the autograd graph shallow; tract IRs decay fast
optimizer = torch.optim.Adam([raw], lr=LR)

best_loss = torch.full((T,), float("inf"), device=device)
best_raw_per_t = torch.zeros(T, n_trainable, device=device)
best_pred_per_t = torch.zeros(T, SAMPLES_PER_FRAME, device=device)

loss_history: list[torch.Tensor] = []  # each entry [T, B]
phase_t = {"fwd": 0.0, "loss": 0.0, "bwd": 0.0, "opt": 0.0}

def reconstruct_best():
    winner_raw = best_raw_per_t  # [T, n_trainable]
    winner_synth = to_synth_params(winner_raw).unsqueeze(0)  # [1, T, 13]

    with torch.no_grad():
        stitched = pink_trombone_ola(
            winner_synth, seed=0, control_rate=CONTROL_RATE, ir_length=IR_LENGTH
        )  # [1, T*hop]

    concat_independent = best_pred_per_t.reshape(1, -1)  # [1, T*hop]
    print(f"stitched: {tuple(stitched.shape)}; concat_independent: {tuple(concat_independent.shape)}")
    
    return concat_independent[0]

pbar = tqdm(range(N_STEPS), desc="per-frame fit")
for step in pbar:
    _sync(); t0 = time.perf_counter()
    optimizer.zero_grad()
    raw_flat = raw.reshape(T * B, 1, n_trainable)
    synth_params = to_synth_params(raw_flat)  # [T*B, 1, 13]
    pred = pink_trombone_ola(
        synth_params, seed=0, control_rate=CONTROL_RATE, ir_length=IR_LENGTH
    )  # [T*B, samples_per_frame]
    _sync(); t1 = time.perf_counter()

    losses_flat = per_frame_loss(pred, tgt_rep)  # [T*B]
    losses_per = losses_flat.reshape(T, B)
    loss = losses_per.mean()
    _sync(); t2 = time.perf_counter()

    loss.backward()
    _sync(); t3 = time.perf_counter()

    optimizer.step()
    _sync(); t4 = time.perf_counter()

    with torch.no_grad():
        step_best, step_best_b = losses_per.min(dim=1)  # [T]
        improved = step_best < best_loss
        if improved.any():
            t_imp = improved.nonzero(as_tuple=True)[0]
            b_imp = step_best_b[t_imp]
            best_loss[t_imp] = step_best[t_imp]
            best_raw_per_t[t_imp] = raw.detach()[t_imp, b_imp]
            best_pred_per_t[t_imp] = pred.detach().reshape(T, B, SAMPLES_PER_FRAME)[t_imp, b_imp]

    phase_t["fwd"]  += t1 - t0
    phase_t["loss"] += t2 - t1
    phase_t["bwd"]  += t3 - t2
    phase_t["opt"]  += t4 - t3

    loss_history.append(losses_per.detach().clone())
    pbar.set_postfix(
        mean=f"{losses_per.mean().item():.4f}",
        best_mean=f"{best_loss.mean().item():.4f}",
        best_max=f"{best_loss.max().item():.4f}",
        step_s=f"{t4 - t0:.2f}",
    )

    if step % 10 == 0:
        recon = reconstruct_best()
        display(Audio(recon.cpu().numpy(), rate=SAMPLE_RATE))

loss_history_t = torch.stack(loss_history).cpu().numpy()  # [N_STEPS, T, B]

print(
    f"\nbest-per-frame loss: min={best_loss.min().item():.4f}, "
    f"mean={best_loss.mean().item():.4f}, max={best_loss.max().item():.4f}"
)

total = sum(phase_t.values())
print(f"\n{'phase':<8} {'total (s)':>10} {'ms/step':>10} {'%':>6}")
for k, v in phase_t.items():
    print(f"{k:<8} {v:10.2f} {v / N_STEPS * 1000:10.1f} {v / total * 100:5.1f}%")
print(f"{'TOTAL':<8} {total:10.2f} {total / N_STEPS * 1000:10.1f} {100.0:5.1f}%")

## 5. Stitch winners and resynth

Two reconstructions:
1. **Stitched (continuous OLA)** — winning params fed to `pink_trombone_ola` over the full T at once. Glottis phase is continuous across frames.
2. **Concatenated independents** — the per-frame predicted audio chunks (each from a phase-zero synth) glued head-to-tail. Likely clicks at frame boundaries; useful as a sanity baseline for what the optimizer actually scored.

In [ ]:
# winner_raw = best_raw_per_t  # [T, n_trainable]
# winner_synth = to_synth_params(winner_raw).unsqueeze(0)  # [1, T, 13]

# with torch.no_grad():
#     stitched = pink_trombone_ola(
#         winner_synth, seed=0, control_rate=CONTROL_RATE, ir_length=IR_LENGTH
#     )  # [1, T*hop]

# concat_independent = best_pred_per_t.reshape(1, -1)  # [1, T*hop]
# print(f"stitched: {tuple(stitched.shape)}; concat_independent: {tuple(concat_independent.shape)}")

concat_independent = reconstruct_best()

## 6. Listen

In [ ]:
def norm(x: torch.Tensor) -> np.ndarray:
    x = x.cpu().numpy()
    peak = np.abs(x).max()
    return x / peak * 0.9 if peak > 1e-6 else x

print("Target:")
display(Audio(norm(target[0]), rate=SAMPLE_RATE))

print("\nStitched fit (continuous-state OLA on winning params):")
display(Audio(norm(stitched[0]), rate=SAMPLE_RATE))

print("\nPer-frame independent winners concatenated (no IR continuity):")
display(Audio(norm(concat_independent), rate=SAMPLE_RATE))

## 7. Loss curves and per-frame final loss

In [ ]:
mean_loss_per_step = loss_history_t.mean(axis=2)  # [N_STEPS, T]
best_loss_np = best_loss.cpu().numpy()
best_t = int(best_loss_np.argmin())
worst_t = int(best_loss_np.argmax())

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "loss per step (one line per frame, mean over B)",
        "best-over-steps loss per frame",
    ),
)
for t_idx in range(T):
    fig.add_trace(go.Scatter(
        y=mean_loss_per_step[:, t_idx],
        mode="lines",
        line=dict(width=1, color="rgba(120,120,120,0.4)"),
        showlegend=False,
        hoverinfo="skip",
    ), row=1, col=1)
fig.add_trace(go.Scatter(
    y=mean_loss_per_step[:, best_t], mode="lines",
    line=dict(width=2.5, color="green"),
    name=f"best frame t={best_t} (L={best_loss_np[best_t]:.3f})",
), row=1, col=1)
fig.add_trace(go.Scatter(
    y=mean_loss_per_step[:, worst_t], mode="lines",
    line=dict(width=2.5, color="red"),
    name=f"worst frame t={worst_t} (L={best_loss_np[worst_t]:.3f})",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=np.arange(T), y=best_loss_np, showlegend=False,
), row=1, col=2)
fig.update_xaxes(title_text="step", row=1, col=1)
fig.update_xaxes(title_text="frame index", row=1, col=2)
fig.update_yaxes(title_text="loss", type="log", row=1, col=1)
fig.update_yaxes(title_text="loss", row=1, col=2)
fig.update_layout(
    template="plotly_white", width=1100, height=400,
    title=f"per-frame fit — {T} frames × {B} inits × {N_STEPS} steps",
)
fig.show()

## 8. Fitted parameter trajectories

In [ ]:
show_names = ["frequency", "tenseness", "tongueIndex", "tongueDiameter", "loudness",
              "constrictionIndex", "constrictionDiameter"]
frames_t = np.arange(T) * SAMPLES_PER_FRAME / SAMPLE_RATE

fig = go.Figure()
for name in show_names:
    i = PARAM_NAMES.index(name)
    lo, hi = PARAM_SPEC[name]
    sig = (winner_synth[0, :, i].detach().cpu().numpy() - lo) / (hi - lo)
    fig.add_trace(go.Scatter(x=frames_t, y=sig, mode="lines+markers", name=name))
fig.update_layout(
    title="Fitted parameter trajectories (per-frame argmin over B) — sigmoid(raw)",
    xaxis_title="time (s)",
    yaxis_title="sigmoid(raw) ∈ [0, 1]",
    yaxis=dict(range=[0, 1]),
    template="plotly_white", width=900, height=450,
)
fig.show()

## 9. Spectrogram comparison

In [ ]:
N_FFT_MEL = 2048
HOP_MEL = N_FFT_MEL // 4
N_MELS = 80
F_MIN_M, F_MAX_M = 0.0, 2000.0
_win_mel = torch.hann_window(N_FFT_MEL, device=device)
_mel_fb = torch.from_numpy(
    librosa.filters.mel(sr=SAMPLE_RATE, n_fft=N_FFT_MEL, n_mels=N_MELS, fmin=F_MIN_M, fmax=F_MAX_M)
).to(device)
mel_centers = librosa.mel_frequencies(n_mels=N_MELS, fmin=F_MIN_M, fmax=F_MAX_M)

def _logmel(x: torch.Tensor) -> np.ndarray:
    s = torch.stft(x.unsqueeze(0), N_FFT_MEL, HOP_MEL, window=_win_mel, return_complex=True).abs()
    m = _mel_fb @ s[0]
    return torch.log1p(m).cpu().numpy()

mel_target = _logmel(target[0])
mel_fit = _logmel(stitched[0])
times = np.arange(mel_target.shape[1]) * HOP_MEL / SAMPLE_RATE
zmax = float(max(mel_target.max(), mel_fit.max()))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("target", "stitched fit"))
for col, spec in [(1, mel_target), (2, mel_fit)]:
    fig.add_trace(go.Heatmap(
        z=spec, x=times, y=mel_centers, zmin=0, zmax=zmax,
        colorscale="Viridis", showscale=(col == 2),
    ), row=1, col=col)
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="mel center freq (Hz)", row=1, col=1)
fig.update_layout(
    title=f"log-mel spectrogram (n_mels={N_MELS}, n_fft={N_FFT_MEL})",
    template="plotly_white", width=1000, height=400,
)
fig.show()